# DAT535 Data Exploration: Option 1 - Weather & Electricity Prices in Norway

This notebook tests data sources, APIs, historical coverage, update rates, volume, and suitability for the **DAT535 Medallion Architecture** project.

## 1. Project Concept
- **Bronze Layer**: Ingest raw hourly electricity spot prices (`hvakosterstrommen.no`) across Norway's 5 bidding zones (`NO1` Oslo, `NO2` Stavanger, `NO3` Trondheim, `NO4` Tromsø, `NO5` Bergen) and hourly historical weather observations (Open-Meteo / MET Frost API).
- **Silver Layer**: Standardize timezone encodings (UTC / CET), handle daylight saving transitions (23h/25h days), impute missing weather readings, deduplicate records, and execute spatial-temporal joins (joining weather parameters with electricity price zones).
- **Gold Layer**: Build business analytics: daily/weekly average prices by zone, price spread across zones, heating degree days (HDD) correlation with spot price spikes, and weather severity indices.


In [1]:
import urllib.request
import json
import pandas as pd
import datetime

# 1. Fetch Electricity Prices for NO2 (Stavanger/South) & NO1 (Oslo)
print('Fetching electricity prices from hvakosterstrommen.no...')
date_str = '2023/12-01'
zones = ['NO1', 'NO2', 'NO3', 'NO4', 'NO5']
all_prices = []

for z in zones:
    url = f'https://www.hvakosterstrommen.no/api/v1/prices/{date_str}_{z}.json'
    req = urllib.request.Request(url, headers={'User-Agent': 'DAT535-Explorer'})
    try:
        with urllib.request.urlopen(req) as resp:
            data = json.loads(resp.read().decode('utf-8'))
            for item in data:
                item['zone'] = z
                all_prices.append(item)
    except Exception as e:
        print(f'Error fetching zone {z}:', e)

df_prices = pd.DataFrame(all_prices)
print(f'Total price records fetched for 1 day across 5 zones: {len(df_prices)}')
display(df_prices.head())


Fetching electricity prices from hvakosterstrommen.no...


Total price records fetched for 1 day across 5 zones: 120


,NOK_per_kWh,EUR_per_kWh,EXR,time_start,time_end,zone
0,1.21059,0.10350,11.6965,2023-12-01T00:00:00+01:00,2023-12-01T01:00:00+01:00,NO1
1,1.16439,0.09955,11.6965,2023-12-01T01:00:00+01:00,2023-12-01T02:00:00+01:00,NO1
2,1.12555,0.09623,11.6965,2023-12-01T02:00:00+01:00,2023-12-01T03:00:00+01:00,NO1
3,1.08894,0.09310,11.6965,2023-12-01T03:00:00+01:00,2023-12-01T04:00:00+01:00,NO1
4,1.09105,0.09328,11.6965,2023-12-01T04:00:00+01:00,2023-12-01T05:00:00+01:00,NO1


In [2]:
# 2. Fetch Weather Data for Stavanger (NO2) and Oslo (NO1)
print('Fetching historical weather from Open-Meteo API...')
locations = {
    'NO1_Oslo': (59.9139, 10.7522),
    'NO2_Stavanger': (58.9701, 5.7331),
    'NO3_Trondheim': (63.4305, 10.3951),
    'NO4_Tromso': (69.6492, 18.9553),
    'NO5_Bergen': (60.3929, 5.3241)
}

start_date = '2023-12-01'
end_date = '2023-12-31'
weather_records = []

for loc_name, (lat, lon) in locations.items():
    url = f'https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start_date}&end_date={end_date}&hourly=temperature_2m,precipitation,rain,snowfall,wind_speed_10m'
    req = urllib.request.Request(url, headers={'User-Agent': 'DAT535-Explorer'})
    try:
        with urllib.request.urlopen(req) as resp:
            data = json.loads(resp.read().decode('utf-8'))['hourly']
            df_loc = pd.DataFrame(data)
            df_loc['location'] = loc_name
            weather_records.append(df_loc)
    except Exception as e:
        print(f'Error fetching weather for {loc_name}:', e)

df_weather_all = pd.concat(weather_records, ignore_index=True)
print(f'Total weather records for 1 month across 5 regions: {len(df_weather_all)}')
display(df_weather_all.head())


Fetching historical weather from Open-Meteo API...


Total weather records for 1 month across 5 regions: 3720


,time,temperature_2m,precipitation,rain,snowfall,wind_speed_10m,location
0,2023-12-01T00:00,-5.8,0.0,0.0,0.0,7.6,NO1_Oslo
1,2023-12-01T01:00,-6.0,0.0,0.0,0.0,7.8,NO1_Oslo
2,2023-12-01T02:00,-6.4,0.0,0.0,0.0,7.4,NO1_Oslo
3,2023-12-01T03:00,-6.7,0.0,0.0,0.0,8.0,NO1_Oslo
4,2023-12-01T04:00,-6.9,0.0,0.0,0.0,8.1,NO1_Oslo


## 2. Technical Evaluation Summary (Option 1)

| Dimension | Details |
| :--- | :--- |
| **Data Availability** | High. APIs are completely free, open, and responsive. |
| **API Endpoint** | `hvakosterstrommen.no` (Prices) & `archive-api.open-meteo.com` (Weather) |
| **History Range** | Electricity prices: Nov 2021 – Present (3+ years). Weather: 1940 – Present (80+ years). |
| **Update Frequency** | Daily (Day-ahead prices updated at 13:00 CET every day; weather updated daily). |
| **Data Volume** | ~43,800 rows/year for 5 price zones + ~43,800 rows/year for 5 weather regions. 3 years of data = ~260,000+ hourly rows. |
| **Spark Suitability** | Perfect fit for PySpark batch pipelines with partitioned Bronze storage by `year`/`month`/`zone`. |


## 3. Comprehensive Data Fields & Additional Sources

### A. Electricity Data Sources & Fields
1. **`hvakosterstrommen.no` API (Used in Quick Demo)**:
   - **Fields**: `time_start`, `time_end`, `NOK_per_kWh`, `EUR_per_kWh`, `EXR` (EUR/NOK Exchange Rate).
   - **History**: November 2021 – Present (3.5+ years).
   - **Ease of Access**: Extremely easy (Direct public REST endpoint, no registration).

2. **ENTSO-E Transparency Platform (Official European Grid API)**:
   - **Fields**: 
     - Spot Day-Ahead Prices (`EUR/MWh`).
     - **Actual Total Load / Consumption** per zone (`MW`).
     - Day-Ahead Total Load Forecast (`MW`).
     - **Generation by Type** (Hydro Water Reservoir, Hydro Run-of-River, Wind Onshore, Solar, Thermal) per zone (`MW`).
     - **Cross-Border Physical Flows** (Import/Export between price zones and Denmark/Germany/UK) (`MW`).
   - **History**: 2015 – Present (10+ years!).
   - **Ease of Access**: Free REST API key registration via `transparency.entsoe.eu`.

3. **Elhub Open Data (`elhub.no`)**:
   - **Fields**: Hourly aggregated electricity grid consumption by Norwegian bidding zone, municipality, and consumption group (Household, Commercial, Industry).
   - **History**: 2019 – Present.

### B. Weather Data Sources & Rich Fields
1. **Open-Meteo Historical Weather API**:
   - **Available Fields**: `temperature_2m`, `apparent_temperature` (feels like), `relative_humidity_2m`, `dew_point_2m`, `precipitation`, `rain`, `snowfall`, `snow_depth`, `weather_code` (WMO code), `surface_pressure`, `cloud_cover` (low, mid, high), `wind_speed_10m`, `wind_direction_10m`, `wind_gusts_10m`, `shortwave_radiation_instant` (solar power potential), `direct_normal_irradiance`.
   - **History**: 1940 – Present (80+ years!).
   - **Ease of Access**: Extremely easy (Free REST API, no key required).

2. **MET Norway Frost API (`frost.met.no`)**:
   - **Fields**: Sensor observations from official Norwegian physical weather stations (air temp, sea temp, snow depth, wind gusts, atmospheric pressure).
   - **History**: Up to 100 years.

---

## 4. Data Volume & File Size Estimates (MB / GB)

### How easy is it to get ALL history?
- **Fetching History**: Very easy. A simple Python script with a loop over years/months can download the entire 3.5 years of electricity prices + 10 years of weather/generation data in under **2-3 minutes**.

### Data Size breakdown:
| Dataset Scope | Time Period | Total Rows | Raw JSON / CSV Size | Parquet Storage (Bronze/Silver) |
| :--- | :--- | :--- | :--- | :--- |
| **Electricity Spot Prices (5 Zones)** | Nov 2021 – Present (~3.5 yrs) | ~170,000 rows | ~22 MB | **~3.5 MB** |
| **Weather Data (5 Regions, 18 fields)** | 2015 – Present (10 yrs) | ~438,000 rows | ~85 MB | **~12 MB** |
| **ENTSO-E Grid Load & Hydro Generation** | 2015 – Present (10 yrs) | ~1,500,000 rows | ~250 MB | **~35 MB** |
| **FULL COMBINED PROJECT DATASET** | **Multi-Year History** | **~2.1 Million rows** | **~350–400 MB** | **~50 MB** |

> **In-Memory Size in PySpark**: When loaded into PySpark memory or uncompressed DataFrames, this dataset expands to **~1.2 GB – 1.8 GB**, making it large enough to demonstrate PySpark distributed processing, shufflings, partition tuning, and joins without crashing a local machine!

In [3]:
# Test fetching 20+ rich weather fields for Stavanger to demonstrate field breadth
import urllib.request
import json
import pandas as pd

rich_fields = [
    'temperature_2m', 'apparent_temperature', 'relative_humidity_2m', 'dew_point_2m',
    'precipitation', 'rain', 'snowfall', 'snow_depth', 'weather_code', 'surface_pressure',
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
    'shortwave_radiation_instant', 'direct_normal_irradiance'
]
fields_str = ','.join(rich_fields)
url = f'https://archive-api.open-meteo.com/v1/archive?latitude=58.9701&longitude=5.7331&start_date=2023-01-01&end_date=2023-01-07&hourly={fields_str}'

req = urllib.request.Request(url, headers={'User-Agent': 'DAT535-Explorer'})
with urllib.request.urlopen(req) as resp:
    data = json.loads(resp.read().decode('utf-8'))['hourly']
    df_rich = pd.DataFrame(data)

print(f'Retrieved dataset with {df_rich.shape[1]} columns and {df_rich.shape[0]} hourly rows!')
display(df_rich.head())


Retrieved dataset with 20 columns and 168 hourly rows!


,time,temperature_2m,apparent_temperature,relative_humidity_2m,dew_point_2m,precipitation,rain,snowfall,snow_depth,weather_code,surface_pressure,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_high,wind_speed_10m,wind_direction_10m,wind_gusts_10m,shortwave_radiation_instant,direct_normal_irradiance
0,2023-01-01T00:00,1.9,-1.5,88,0.2,0.0,0.0,0.0,0.0,2,998.4,60,4,1,64,8.7,38,11.2,0.0,0.0
1,2023-01-01T01:00,0.9,-2.2,89,-0.7,0.0,0.0,0.0,0.0,3,999.0,100,5,6,100,5.8,68,13.7,0.0,0.0
2,2023-01-01T02:00,0.9,-2.3,89,-0.7,0.0,0.0,0.0,0.0,3,998.9,81,10,0,79,6.5,71,11.5,0.0,0.0
3,2023-01-01T03:00,-0.2,-3.2,89,-1.8,0.0,0.0,0.0,0.0,3,999.6,99,16,1,99,4.3,95,10.1,0.0,0.0
4,2023-01-01T04:00,0.1,-3.3,91,-1.3,0.0,0.0,0.0,0.0,2,999.1,71,25,3,59,7.5,55,10.1,0.0,0.0


--- 
## 5. MET Norway Frost API (Sola Weather Station SN44520)

This section allows you to query official weather observations directly from **Stavanger Airport - Sola (Station ID: `SN44520`)** using your free MET Frost API `client_id`.

### Instructions:
1. Replace `'YOUR_CLIENT_ID_HERE'` in the cell below with your actual Client ID.
2. Run the cell to fetch:
   - **Part A**: The complete catalog of **all available weather elements/fields** measured at Sola.
   - **Part B**: A sample hourly observation DataFrame containing all measured fields.


In [4]:
# =========================================================================
# MET NORWAY FROST API - STAVANGER / SOLA WEATHER STATION (SN44520)
# =========================================================================
import requests
import pandas as pd
import json

# 🔑 INSERT YOUR MET FROST CLIENT ID HERE:
CLIENT_ID = '6b0b9a38-4fdb-40a6-9192-7a9bb7e72873'

STATION_ID = 'SN44520'  # Stavanger Airport - Sola station

if CLIENT_ID == 'YOUR_CLIENT_ID_HERE':
    print('⚠️ Please replace YOUR_CLIENT_ID_HERE with your actual MET Frost Client ID.')
else:
    print(f'Connecting to MET Frost API for station {STATION_ID}...')
    
    # ---------------------------------------------------------------------
    # PART A: Discover ALL available weather elements/fields at Sola station
    # ---------------------------------------------------------------------
    ts_url = 'https://frost.met.no/observations/availableTimeSeries/v0.jsonld'
    res_ts = requests.get(ts_url, params={'sources': STATION_ID}, auth=(CLIENT_ID, ''))
    
    if res_ts.status_code == 200:
        elements_data = res_ts.json().get('data', [])
        element_list = []
        for item in elements_data:
            element_list.append({
                'elementId': item.get('elementId'),
                'unit': item.get('unit'),
                'timeOffset': item.get('timeOffset'),
                'timeResolution': item.get('timeResolution'),
                'performanceCategory': item.get('performanceCategory')
            })
        df_elements = pd.DataFrame(element_list).drop_duplicates(subset=['elementId'])
        print(f'✅ Found {len(df_elements)} unique weather measurement fields at Sola (SN44520)!')
        display(df_elements)
    else:
        print(f'❌ Error discovering elements: HTTP {res_ts.status_code}', res_ts.text)

    # ---------------------------------------------------------------------
    # PART B: Fetch observation data for a sample week across all fields
    # ---------------------------------------------------------------------
    obs_url = 'https://frost.met.no/observations/v0.jsonld'
    params = {
        'sources': STATION_ID,
        'referencetime': '2024-01-01/2024-01-07',
        'elements': 'air_temperature,sum(precipitation_amount PT1H),wind_speed,relative_humidity,surface_snow_thickness,air_pressure_at_sea_level'
    }
    res_obs = requests.get(obs_url, params=params, auth=(CLIENT_ID, ''))
    
    if res_obs.status_code == 200:
        raw_obs = res_obs.json().get('data', [])
        records = []
        for entry in raw_obs:
            row = {'referenceTime': entry.get('referenceTime'), 'sourceId': entry.get('sourceId')}
            for obs in entry.get('observations', []):
                row[obs.get('elementId')] = obs.get('value')
            records.append(row)
        
        df_sola_obs = pd.DataFrame(records)
        print(f'\n✅ Successfully fetched {len(df_sola_obs)} observation rows!')
        print(f'Columns returned ({len(df_sola_obs.columns)} total): {list(df_sola_obs.columns)}')
        display(df_sola_obs.head())
    else:
        print(f'❌ Error fetching observations: HTTP {res_obs.status_code}', res_obs.text)


Connecting to MET Frost API for station SN44520...


✅ Found 43 unique weather measurement fields at Sola (SN44520)!


,elementId,unit,timeOffset,timeResolution,performanceCategory
0,state_of_ground,code,PT6H,P1D,C
1,sum(precipitation_amount_anomaly P6M 1961_1990),percent,PT6H,P6M,C
2,sum(precipitation_amount_anomaly P3M 1961_1990),percent,PT6H,P3M,C
3,sum(precipitation_amount P6M),mm,PT6H,P6M,C
4,max(sum(precipitation_amount P1D) P1M),mm,PT6H,P1M,C
5,best_estimate_sum(precipitation_amount_anomaly...,percent,PT6H,P3M,C
6,over_time(precipitation_type P1D),code,PT6H,P1D,C
7,best_estimate_sum(precipitation_amount_anomaly...,percent,PT6H,P1M,C
8,day_of_year_for_birch_leafing,day number,PT0H,P1Y,C
9,best_estimate_sum(precipitation_amount_anomaly...,percent,PT6H,P1Y,C



✅ Successfully fetched 6 observation rows!
Columns returned (3 total): ['referenceTime', 'sourceId', 'surface_snow_thickness']


,referenceTime,sourceId,surface_snow_thickness
0,2024-01-01T00:00:00.000Z,SN44520:0,0
1,2024-01-02T00:00:00.000Z,SN44520:0,0
2,2024-01-03T00:00:00.000Z,SN44520:0,0
3,2024-01-04T00:00:00.000Z,SN44520:0,0
4,2024-01-05T00:00:00.000Z,SN44520:0,0
